# **BLAST MASTER GENERATOR**

## 🔍 **Step 0: Load the Scheduled Block Model**

Load the scheduled block model file containing spatial coordinates, economic values, classification, and scheduling outputs (e.g., phase, bench, pushback, period).

This file represents the result of pit shell filtering, mining phase assignment, and production scheduling.

The notebook supports `.csv`, `.json`, and `.parquet` formats.

Also import the necessary libraries and set the file path to access the data.

In [423]:
# ======================================================================
# 📦 STEP 0: LOAD THE SCHEDULED BLOCK MODEL
# ======================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px
import os
import math
from scipy.spatial import ConvexHull
from matplotlib.patches import Rectangle

# --- File path setup ---
file_path = "../data/"
filename_base = "scheduled_block_model"
file_type = ".csv"  # Options: .parquet, .csv, .json
file = os.path.join(file_path, f"{filename_base}{file_type}")
print(f"File path: {file}")

# --- Load Scheduled Block Model ---
if file.endswith(".csv"):
    block_df = pd.read_csv(file)
elif file.endswith(".json"):
    block_df = pd.read_json(file)
elif file.endswith(".parquet"):
    block_df = pd.read_parquet(file)
else:
    raise ValueError("Unsupported file format")

# --- Preview ---
print(f"Loaded {len(block_df):,} blocks.")
display(block_df.head())


File path: ../data/scheduled_block_model.csv
Loaded 32,711 blocks.


,x,y,z,concentric_zone,lith_code,processing_route,Au_grade,Cu_grade,fault_zone_category,density,...,pushback_x,pushback_y,period,original_period,period_str,period_frame,Au_oz,Cu_tonnes,is_ore,is_waste
0,1662.5,537.5,7.5,Background,GRAN,Bioleach,0.074268,0.041017,none,2.653928,...,1660,520,78,80,78,78,59.408748,10.205285,False,True
1,1637.5,537.5,7.5,Background,GRAN,Bioleach,0.076331,0.039876,none,2.660451,...,1620,520,77,79,77,77,61.209411,9.945880,False,True
2,1287.5,812.5,157.5,Background,GRAN,Bioleach,0.097918,0.033620,none,2.780438,...,1280,800,77,79,77,77,82.061119,8.763619,False,True
3,2487.5,987.5,157.5,Background,GRAN,Bioleach,0.085950,0.029023,none,2.853997,...,2480,980,77,79,77,77,73.937349,7.765412,False,True
4,2562.5,1262.5,157.5,Background,GRAN,Bioleach,0.083280,0.033246,none,2.902946,...,2560,1260,77,79,77,77,72.868546,9.048001,False,True


## ⚙️ **Step 1: Define User Parameters**

Define key parameters used throughout the blast master generation notebook.

These include operational constraints (bench height, ramp gradient), equipment assumptions (e.g., face shovel, truck type), and blast design logic.

The goal is to generate blast patterns that align with operational fleet productivity, practical drill & blast cycles, and realistic pit development strategies for a medium to large-scale open pit mine.

In [424]:
# ======================================================================
# 📦 STEP 1: DEFINE USER PARAMETERS ###
# ======================================================================
import numpy as np

# ======================================================================
# 📋 GENERAL PARAMETES
# ======================================================================

bench_height = 15  					# metres
ramp_gradient = 1 / 10  			# 10% gradient
inter_ramp_angle_deg = 45
catch_berm_interval = 30  			# vertical metres
catch_berm_width = 5  				# metres
min_mining_width = 20  				# metres

# ======================================================================
# 📋 EQUIPMENT SPECIFICATIONS
# ======================================================================

truck_model = "Caterpillar 793"
shovel_model = "Hitachi EX8000"
equipment_specs = {
    "Caterpillar 793": {
        "payload_t": 231,      		# tonnes
        "speed_loaded_kph": 40,
        "speed_empty_kph": 60,
        "width_m": 7.6,
        "length_m": 15.5,
        "min_operating_width_m": 30 # min clear width including safety
    },
    "Hitachi EX8000": {
        "bucket_capacity_m3": 40,
        "fill_factor": 0.9,
        "density_t_m3": 2.5,  		# assumed average material density
        "dig_rate_tph": 4500,
        "reach_m": 20
    }
}

min_truck_width = np.ceil(equipment_specs['Caterpillar 793']['width_m'])

# ======================================================================
# 📋 BLAST PATTERN GRID PARAMETERS
# ======================================================================

pit_development_direction = "north_to_south"    # other options: "east_to_west", "from_center"
blast_pattern_size_x = 100  			        # metres (typical burden)
blast_pattern_size_y = 350  			        # metres (typical spacing)
pattern_overlap_buffer = 0.5  		            # metres, optional overlap for pattern continuity

min_trim_width = 3 * min_truck_width            # metres (minimum pattern width for trim blasts)

# ======================================================================
# 📋 PRODUCTIVITY ASSUMPTIONS
# ======================================================================

# Aim for each blast to feed the shovel for ~6 shifts
target_dig_hours_per_pattern = 72  	# 6 shifts × 12 hours
shovel_dig_rate_tph = 4500  		# tonnes per hour (typical for EX8000)
target_pattern_tonnage = target_dig_hours_per_pattern * shovel_dig_rate_tph

# ======================================================================
# 📋 DEVELOPMENT STRATEGY
# ======================================================================

early_years_slower = True  			# slower progress during pre-strip
ramp_up_year = 2  					# full production ramp-up expected by year 2–3
max_pattern_tonnage = target_pattern_tonnage * 2  # cap large patterns

# ======================================================================
# 📋 VISUALISATION / DEBUGGING OPTIONS
# ======================================================================

preview_bench = 1020  				# Example RL to preview pattern generation

In [425]:
# Print the unique RL values in the block model sorted by z
print("Unique RL values in the block model sorted by z:")
block_df = block_df.sort_values(by='z')
print(block_df['z'].unique())  # Check the RL values in the block model

block_df.head()

Unique RL values in the block model sorted by z:
[  7.5 112.5 127.5 142.5 157.5 172.5 187.5 202.5 217.5 232.5 247.5 262.5
 277.5 292.5 307.5 322.5 337.5 352.5 367.5 382.5 397.5 412.5 427.5 442.5
 457.5 472.5 487.5 502.5 517.5 532.5 547.5 562.5 577.5 592.5 607.5 622.5]


,x,y,z,concentric_zone,lith_code,processing_route,Au_grade,Cu_grade,fault_zone_category,density,...,pushback_x,pushback_y,period,original_period,period_str,period_frame,Au_oz,Cu_tonnes,is_ore,is_waste
737,1737.5,637.5,7.5,Background,GRAN,Bioleach,0.082649,0.039834,none,2.657966,...,1720,620,74,76,74,74,66.213871,9.926075,False,True
738,1762.5,637.5,7.5,Background,GRAN,Bioleach,0.085117,0.038920,none,2.650107,...,1760,620,74,76,74,74,67.989635,9.669698,False,True
739,1987.5,637.5,7.5,Background,GRAN,Bioleach,0.107065,0.028021,halo,2.603479,...,1980,620,74,76,74,74,84.016497,6.839235,False,True
724,1812.5,637.5,7.5,Background,GRAN,Bioleach,0.092525,0.035612,none,2.635217,...,1800,620,74,76,74,74,73.491514,8.797982,False,True
725,1837.5,637.5,7.5,Background,GRAN,Bioleach,0.097801,0.034161,none,2.628419,...,1820,620,74,76,74,74,77.481836,8.417671,False,True


## 📐 **Step 2:  Bench Assignment & Edge Detection
* Assign each block to a bench level using its z value and bench_height.
* Filter out unmined or inaccessible blocks (has_access == True and within_shell == True).

In [426]:
# ======================================================================
# 🪜 STEP 2: BENCH ASSIGNMENT & EDGE DETECTION
# ======================================================================

bench_origin_z = 0

# Assign benches
block_df["bench_level"] = ((block_df["z"] - bench_origin_z) // bench_height).astype(int)
block_df["bench_rl"] = block_df["bench_level"] * bench_height + bench_origin_z + (bench_height / 2)

# Filter accessible, mineable blocks
mined_blocks = block_df[(block_df["within_shell"]) & (block_df["has_access"])].copy()

# Add edge flags (used for trim logic)
x_limits = mined_blocks["x"].quantile([0.01, 0.99])
y_limits = mined_blocks["y"].quantile([0.01, 0.99])

mined_blocks["is_edge_block"] = (
    (mined_blocks["x"] <= x_limits.iloc[0]) |
    (mined_blocks["x"] >= x_limits.iloc[1]) |
    (mined_blocks["y"] <= y_limits.iloc[0]) |
    (mined_blocks["y"] >= y_limits.iloc[1])
)

# Bench Summary
bench_summary = (
    mined_blocks.groupby("bench_rl")
    .agg(num_blocks=("block_id", "count"), total_tonnes=("tonnes", "sum"))
    .reset_index()
)
bench_summary["total_tonnes"] = (bench_summary["total_tonnes"] / 1e6).round(2)
print("Total tonnes per bench (millions):")
print(bench_summary.to_string(index=False))


Total tonnes per bench (millions):
 bench_rl  num_blocks  total_tonnes
      7.5         396          9.85
    112.5        2241         57.01
    127.5        1994         50.67
    142.5        2073         52.83
    157.5        1998         50.96
    172.5        1922         49.11
    187.5        1700         43.34
    202.5        1720         44.01
    217.5        1664         42.61
    232.5        1434         36.68
    247.5        1468         37.64
    262.5        1280         32.80
    277.5        1242         31.87
    292.5        1168         30.03
    307.5        1112         28.62
    322.5        1016         26.19
    337.5         948         24.47
    352.5         864         22.34
    367.5         812         21.02
    382.5         712         18.47
    397.5         684         17.76
    412.5         596         15.51
    427.5         560         14.59
    442.5         484         12.63
    457.5         448         11.71
    472.5         384        

 ## 🎯 **Step 3: Grid-Based Blast Pattern Grouping**

In [ ]:
# ======================================================================
# 🎯 STEP 3: GRID-BASED BLAST PATTERN GROUPING
# ======================================================================

# Assign grid coordinates
pattern_dx = blast_pattern_size_x
pattern_dy = blast_pattern_size_y

mined_blocks["pattern_col"] = (mined_blocks["x"] // pattern_dx).astype(int)
mined_blocks["pattern_row"] = (mined_blocks["y"] // pattern_dy).astype(int)

mined_blocks["blast_id"] = (
    "B" + mined_blocks["bench_level"].astype(str).str.zfill(2) +
    "_R" + mined_blocks["pattern_row"].astype(str).str.zfill(3) +
    "_C" + mined_blocks["pattern_col"].astype(str).str.zfill(3)
)


#### 🛠️ **Step 3.5: Structured Blast Pattern Assignment - Ramp + Trim + Regular Pattern**



In [ ]:
# ======================================================================
# 🛠️ STEP 3.5: REFINE RAMP AND TRIM PATTERN CLASSIFICATION
# ======================================================================

# --- Step 1: Aggregate block stats per blast pattern ---
blast_bounds = (
    mined_blocks
    .groupby("blast_id")
    .agg(
        min_x=("x", "min"), max_x=("x", "max"),
        min_y=("y", "min"), max_y=("y", "max"),
        min_z=("z", "min"), max_z=("z", "max"),
        bench_rl=("bench_rl", "first")
    )
    .copy()
)

blast_bounds["width_x"] = blast_bounds["max_x"] - blast_bounds["min_x"]
blast_bounds["width_y"] = blast_bounds["max_y"] - blast_bounds["min_y"]
blast_bounds["height_z"] = blast_bounds["max_z"] - blast_bounds["min_z"]

# --- Step 2: Classify pattern type ---
# Trim: Small in either X or Y direction
blast_bounds["is_trim"] = (
    (blast_bounds["width_x"] < min_trim_width) |
    (blast_bounds["width_y"] < min_trim_width)
)

# Ramp: Long + narrow based on 1:10 gradient
ramp_required_length = bench_height * 10 + 3 * min_truck_width
ramp_width = 3 * min_truck_width
blast_bounds["is_ramp"] = (
    ((blast_bounds["width_x"] >= ramp_required_length) & (blast_bounds["width_y"] <= ramp_width)) |
    ((blast_bounds["width_y"] >= ramp_required_length) & (blast_bounds["width_x"] <= ramp_width))
)

# --- Step 3: Assign exclusive pattern type ---
blast_bounds["blast_pattern_type"] = "Regular"
blast_bounds.loc[blast_bounds["is_trim"], "blast_pattern_type"] = "Trim"
blast_bounds.loc[blast_bounds["is_ramp"], "blast_pattern_type"] = "Ramp"

# --- Step 4: Store final mapping ---
blast_pattern_type_map = blast_bounds[["blast_pattern_type", "bench_rl"]].copy()
print("🚦 Pattern Type Breakdown (by blast):")
print(blast_pattern_type_map["blast_pattern_type"].value_counts())

# For debugging
print("Ramp benches:", blast_pattern_type_map[blast_pattern_type_map["blast_pattern_type"] == "Ramp"]["bench_rl"].unique())
print("Trim benches:", blast_pattern_type_map[blast_pattern_type_map["blast_pattern_type"] == "Trim"]["bench_rl"].unique())


## 📦 **Step 4: Create Blast Master Table**

In [ ]:
# ======================================================================
# 📦 STEP 4: CREATE BLAST MASTER TABLE
# ======================================================================

blast_master = (
    mined_blocks
    .groupby("blast_id")
    .agg(
        x_centroid=("x", "mean"),
        y_centroid=("y", "mean"),
        z_centroid=("z", "mean"),
        bench_level=("bench_level", "first"),
        bench_rl=("bench_rl", "first"),
        phase=("phase", lambda x: x.mode().iloc[0]),
        period=("period", lambda x: x.mode().iloc[0]),
        tonnes=("tonnes", "sum"),
        Au_grade=("Au_grade", "mean"),
        Cu_grade=("Cu_grade", "mean"),
        value_per_tonne=("value_per_tonne", "mean"),
        num_blocks=("block_id", "count")
    )
    .reset_index()
)

# --- Join pattern_type from 3.5 ---
blast_master = blast_master.merge(
    blast_pattern_type_map, on=["blast_id", "bench_rl"], how="left"
)

# --- Assign campaign ID ---
blast_master["blast_campaign_id"] = (
    "PH" + blast_master["phase"].astype(str).str.zfill(2) +
    "_P" + blast_master["period"].astype(str).str.zfill(2) +
    "_B" + blast_master["bench_level"].astype(str).str.zfill(2)
)

# --- Check classification ---
print("✅ Pattern types in blast_master:")
print(blast_master["blast_pattern_type"].value_counts())


In [ ]:
# Filter blast_master by blast_pattern_type = Ramp
blast_master_ramp = blast_master[blast_master["pattern_type"] == "Ramp"].copy()

# # Filter blast_master_ramp by bench_rl = preview_bench
# blast_master_ramp_preview = blast_master_ramp[blast_master_ramp["bench_rl"] == preview_bench].copy()

blast_master_ramp

In [ ]:
blast_master.columns

### 📦 **Step 4.1:  Assign Blast Sequence**

In [ ]:
# ======================================================================
# 🔢 STEP 4.1: ASSIGN BLAST SEQUENCE
# ======================================================================

# Consistent column naming
blast_master.rename(columns={"blast_pattern_type": "pattern_type"}, inplace=True)

# 🔁 Ensure pattern_type (Ramp/Trim/Regular) exists in blast_master
if "pattern_type" not in blast_master.columns:
    pattern_map = mined_blocks.drop_duplicates("blast_id")[["blast_id", "blast_pattern_type"]]
    blast_master = blast_master.merge(pattern_map, on="blast_id", how="left")
    blast_master.rename(columns={"blast_pattern_type": "pattern_type"}, inplace=True)

# 🔃 Apply spatial sorting based on pit development direction
if pit_development_direction == "north_to_south":
    sort_order = ["blast_campaign_id", "y_centroid", "x_centroid"]
    ascending_order = [True, True, True]

elif pit_development_direction == "south_to_north":
    sort_order = ["blast_campaign_id", "y_centroid", "x_centroid"]
    ascending_order = [True, False, True]

elif pit_development_direction == "east_to_west":
    sort_order = ["blast_campaign_id", "x_centroid", "y_centroid"]
    ascending_order = [True, False, True]

elif pit_development_direction == "west_to_east":
    sort_order = ["blast_campaign_id", "x_centroid", "y_centroid"]
    ascending_order = [True, True, True]

elif pit_development_direction == "from_center":
    pit_center_x = blast_master["x_centroid"].mean()
    pit_center_y = blast_master["y_centroid"].mean()
    blast_master["distance_from_center"] = np.sqrt(
        (blast_master["x_centroid"] - pit_center_x) ** 2 +
        (blast_master["y_centroid"] - pit_center_y) ** 2
    )
    sort_order = ["blast_campaign_id", "distance_from_center"]
    ascending_order = [True, True]

else:
    raise ValueError(f"Unsupported pit_development_direction: {pit_development_direction}")

# 🧨 Assign sequence number
blast_master["blast_sequence"] = (
    blast_master
    .sort_values(by=sort_order, ascending=ascending_order)
    .groupby("blast_campaign_id")
    .cumcount() + 1
)

# Filter bench_rl to 127.5 for preview
target_rl = 127.5
target_phase = 'Phase 1'

# Filter blast_master for the target bench_rl and phase
blast_master_preview = blast_master[
    (blast_master["bench_rl"] == target_rl) & (blast_master["phase"] == target_phase)
].copy()

# blast_master_preview = blast_master[blast_master["bench_rl"] == target_rl].copy()
len(blast_master_preview)
# blast_master_preview['phase'].unique()
blast_master_preview['pattern_type'].unique()


## ✅ **STEP 5: Visualise Blast Patterns with Ramps, Trims, and Sequence**
This step adds:
* Distinct colouring for Ramp, Trim, and Regular patterns
* Blast sequence numbers overlaid on polygons
* Dropdown selectors to view different phases and benches
#### 📦 **What We’ll Need:**
* blast_master for sequence, centroids, and pattern types
* mined_blocks for individual blocks tied to each blast_id
* The latest logic for identifying is_trim_blast, is_ramp_pattern

In [ ]:
# Filter bench_rl to 127.5 for preview
target_rl = 127.5
target_phase = 'Phase 1'

# Filter blast_master for the target bench_rl and phase
blast_master_preview = blast_master[
    (blast_master["bench_rl"] == target_rl) & (blast_master["phase"] == target_phase)
].copy()

# blast_master_preview = blast_master[blast_master["bench_rl"] == target_rl].copy()
len(blast_master_preview)
# blast_master_preview['phase'].unique()
blast_master_preview['pattern_type'].unique()

In [ ]:
# ======================================================================
# 🧭 STEP 5: INTERACTIVE VISUALISATION – PATTERNS & SEQUENCE
# ======================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.spatial import ConvexHull, QhullError
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.lines as mlines

# Dropdown options
phase_options = sorted(blast_master["phase"].unique())
bench_options = sorted(blast_master["bench_rl"].unique(), reverse=False)

phase_dropdown = widgets.Dropdown(options=phase_options, description="Phase:")
bench_dropdown = widgets.Dropdown(options=bench_options, description="Bench RL:")

def plot_patterns(phase, bench):
    clear_output(wait=True)
    display(widgets.HBox([phase_dropdown, bench_dropdown]))

    # Filter blast_master and mined_blocks for selection
    df_plot = blast_master[
        (blast_master["phase"] == phase) &
        (blast_master["bench_rl"] == bench)
    ]
    blocks = mined_blocks[
        (mined_blocks["phase"] == phase) &
        (mined_blocks["bench_rl"] == bench)
    ]

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.scatter(blocks["x"], blocks["y"], s=5, c="lightgrey", label="Blocks")

    for _, row in df_plot.iterrows():
        pattern_blocks = blocks[blocks["blast_id"] == row["blast_id"]]
        if len(pattern_blocks) < 3:
            continue

        points = pattern_blocks[["x", "y"]].values
        center = points.mean(axis=0)
        scaled_points = center + (points - center) * 1.1  # expand 10%

        try:
            hull = ConvexHull(scaled_points, qhull_options='QJ')
            hull_pts = scaled_points[hull.vertices]

            # 🧠 Pull from pattern_type column (from blast_master)
            ptype = row["pattern_type"].lower()
            if ptype == "ramp":
                edge_color, line_style = "red", "--"
            elif ptype == "trim":
                edge_color, line_style = "blue", ":"
            else:
                edge_color, line_style = "black", "-"

            ax.plot(*hull_pts.T, color=edge_color, linestyle=line_style, linewidth=1.5)

            # Draw sequence number
            ax.text(row["x_centroid"], row["y_centroid"], str(row["blast_sequence"]),
                    fontsize=8, ha="center", va="center", weight="bold")

        except QhullError:
            continue

    ax.set_title(f"Blast Patterns – Phase {phase}, Bench RL {bench}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_aspect("equal")
    ax.grid(True)

    legend_handles = [
        mlines.Line2D([], [], color="red", linestyle="--", label="Ramp"),
        mlines.Line2D([], [], color="blue", linestyle=":", label="Trim"),
        mlines.Line2D([], [], color="black", linestyle="-", label="Regular")
    ]
    ax.legend(handles=legend_handles, loc="upper right")
    plt.tight_layout()
    plt.show()

def update_plot(*args):
    plot_patterns(phase_dropdown.value, bench_dropdown.value)

phase_dropdown.observe(update_plot, names="value")
bench_dropdown.observe(update_plot, names="value")

# Trigger first plot
update_plot()


## 🎯 **Step 3: Create Grid-Based Blast Patterns**
For each bench:
* Divide XY space into regular tiles of size blast_pattern_size_x × blast_pattern_size_y.
* Assign a blast_id as a combination of:
	* Bench level
	* Grid row/column (e.g., B05_R03_C07)
* Compute the polygon centroid, tonnage, average grades, and structural flags.

#### 📌 **Goals:**
1. For each bench:
	* Tile the XY space into rectangular blast zones (e.g., 25 × 25 m).
	* Assign a blast_id to each tile.
2. Associate each mined block with a blast_id based on its XY coordinates.
3. Group by blast_id to compute pattern-level metrics (tonnes, grades, location, etc.).

In [ ]:
# ======================================================================
# 🎯 STEP 3: GRID-BASED BLAST PATTERN GROUPING
# ======================================================================

# First, set up the XY tiling for mined blocks
pattern_dx = blast_pattern_size_x
pattern_dy = blast_pattern_size_y

# Assign pattern bins (row/col) based on origin
mined_blocks["pattern_col"] = ((mined_blocks["x"]) // pattern_dx).astype(int)
mined_blocks["pattern_row"] = ((mined_blocks["y"]) // pattern_dy).astype(int)

# Combine row, col, and bench to create a unique blast_id
mined_blocks["blast_id"] = (
    "B" + mined_blocks["bench_level"].astype(str).str.zfill(2) +
    "_R" + mined_blocks["pattern_row"].astype(str).str.zfill(3) +
    "_C" + mined_blocks["pattern_col"].astype(str).str.zfill(3)
)

blast_id_preview = mined_blocks[mined_blocks["bench_level"] == mined_blocks["bench_level"].max()]
print(f"🧨 Sample blast_ids on upper bench:\n{blast_id_preview['blast_id'].unique()[:10]}")

# ======================================================================
# 🎯 Flag Trim Blasts (based on pattern dimensions)
# ======================================================================

# Define blast pattern bounds (min/max X and Y)
blast_bounds = (
    mined_blocks
    .groupby("blast_id")
    .agg(
        min_x=("x", "min"),
        max_x=("x", "max"),
        min_y=("y", "min"),
        max_y=("y", "max")
    )
)

blast_bounds["width_x"] = blast_bounds["max_x"] - blast_bounds["min_x"]
blast_bounds["width_y"] = blast_bounds["max_y"] - blast_bounds["min_y"]

# A trim blast is one with either dimension below the threshold
blast_bounds["is_trim_blast"] = (
    (blast_bounds["width_x"] < min_trim_width) | 
    (blast_bounds["width_y"] < min_trim_width)
)

# Merge flag back to mined_blocks
mined_blocks = mined_blocks.merge(
    blast_bounds[["is_trim_blast"]], on="blast_id", how="left"
)



### 📊 **Step 3.1: Visual: Blast Grid Layout (Preview)**

In [ ]:
preview_df.columns

In [ ]:
# Count the number of unique blast_ids
unique_blast_ids = mined_blocks["blast_id"].nunique()
print(f"Unique blast_ids: {unique_blast_ids:,}")

In [ ]:
# Plot blast patterns on selected bench
bench_level = int((preview_bench - bench_origin_z) // bench_height)
preview_df = mined_blocks[mined_blocks["bench_level"] == bench_level].copy()

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=preview_df,
    x="x", y="y",
    hue="blast_id",
    palette="tab20",
    s=10,
    legend=False
)
plt.title(f"Blast Pattern Layout for Bench RL {preview_bench}")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)
plt.tight_layout()
plt.show()


## 📦 **Step 4 Overview – Blast Master Table**
This step:
* Aggregates blocks into blast_id-level entries
* Joins metadata: phase, period, bench_level, etc.
* Flags: is_trim_blast, is_edge_blast, is_final_wall (placeholder)
* Tags each pattern with a blast_campaign_id (grouped by phase + period + bench)
* Outputs a CSV file: blast_master_table.csv

In [ ]:
# ======================================================================
# 📦 STEP 4: CREATE BLAST MASTER TABLE
# ======================================================================

# --- Aggregate each blast_id ---
blast_master = (
    mined_blocks
    .groupby("blast_id")
    .agg(
        x_centroid=("x", "mean"),
        y_centroid=("y", "mean"),
        z_centroid=("z", "mean"),
        bench_level=("bench_level", "first"),
        bench_rl=("bench_rl", "first"),
        phase=("phase", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        period=("period", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        tonnes=("tonnes", "sum"),
        Au_grade=("Au_grade", "mean"),
        Cu_grade=("Cu_grade", "mean"),
        value_per_tonne=("value_per_tonne", "mean"),
        num_blocks=("block_id", "count")
    )
    .reset_index()
)

# --- Join edge and trim blast flags ---
edge_blast_ids = preview_df[preview_df["is_edge_block"]]["blast_id"].unique()
trim_blast_ids = preview_df[preview_df["is_trim_blast"]]["blast_id"].unique()

blast_master["is_edge_blast"] = blast_master["blast_id"].isin(edge_blast_ids)
blast_master["is_trim_blast"] = blast_master["blast_id"].isin(trim_blast_ids)

# --- Assign blast campaign ID (Phase + Period + Bench) ---
blast_master["blast_campaign_id"] = (
    "PH" + blast_master["phase"].astype(str).str.zfill(2) +
    "_P" + blast_master["period"].astype(str).str.zfill(2) +
    "_B" + blast_master["bench_level"].astype(str).str.zfill(2)
)

# --- Assign blast sequence number based on pit development direction ---

if pit_development_direction == "north_to_south":
    # Mine from high Y to low Y
    sort_order = ["blast_campaign_id", "y_centroid", "x_centroid"]
    ascending_order = [True, True, True]

elif pit_development_direction == "south_to_north":
    sort_order = ["blast_campaign_id", "y_centroid", "x_centroid"]
    ascending_order = [True, False, True]

elif pit_development_direction == "east_to_west":
    sort_order = ["blast_campaign_id", "x_centroid", "y_centroid"]
    ascending_order = [True, False, True]

elif pit_development_direction == "west_to_east":
    sort_order = ["blast_campaign_id", "x_centroid", "y_centroid"]
    ascending_order = [True, True, True]

elif pit_development_direction == "from_center":
    # Compute distance from center
    pit_center_x = blast_master["x_centroid"].mean()
    pit_center_y = blast_master["y_centroid"].mean()
    blast_master["distance_from_center"] = (
        ((blast_master["x_centroid"] - pit_center_x) ** 2 +
         (blast_master["y_centroid"] - pit_center_y) ** 2) ** 0.5
    )
    sort_order = ["blast_campaign_id", "distance_from_center"]
    ascending_order = [True, True]

else:
    raise ValueError(f"Unsupported pit_development_direction: {pit_development_direction}")

# Apply spatial sequence based on strategy
blast_master["blast_sequence"] = (
    blast_master
    .sort_values(by=sort_order, ascending=ascending_order)
    .groupby("blast_campaign_id")
    .cumcount() + 1
)


# --- Export as CSV ---
blast_master_path = os.path.join(file_path, "blast_master_table.csv")
blast_master.to_csv(blast_master_path, index=False)
print(f"✅ Blast master table saved to: {blast_master_path}")


### 🧭 **Step 4.1: Pit Development Direction: north_to_south**
In this context:
* High Y = north
* Low Y = south
* So we expect mining to progress from top of the plot (high Y) → bottom (low Y)

#### ✅ **What the Plot Shows:**
* Each dot is a blast pattern (blast_id) in a specific campaign (phase + period + bench).
* Color represents the blast sequence (darker = earlier, lighter = later).
* So:
    * Dark dots at the bottom (low Y) = early blasts
    * Light dots at the top (high Y) = later blasts
    * That’s consistent with north_to_south mining, which is exactly what we want.

In [ ]:
blast_master["blast_sequence"] = (
    blast_master
    .sort_values(by=["blast_campaign_id", "y_centroid", "x_centroid"])
    .groupby("blast_campaign_id")
    .cumcount() + 1
)


# 🔍 Visualise one campaign's blast sequence (example)
campaign_to_plot = blast_master["blast_campaign_id"].value_counts().index[0]
campaign_df = blast_master[blast_master["blast_campaign_id"] == campaign_to_plot]

plt.figure(figsize=(12, 10))
sc = plt.scatter(
    campaign_df["x_centroid"], campaign_df["y_centroid"],
    c=campaign_df["blast_sequence"], cmap="viridis", s=80, edgecolor="k"
)
plt.colorbar(sc, label="Blast Sequence")
plt.title(f"Blast Pattern Sequence – {campaign_to_plot}")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)
plt.gca().set_aspect("equal")
plt.tight_layout()
plt.show()


## 📦 **STEP 5: Blast Pattern Visualisation with Ramp Prioritisation**

In [ ]:
# ======================================================================
# 🎯 STEP 5: INTERACTIVE VISUALISATION OF BLAST SEQUENCES
# ======================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import PatchCollection
from scipy.spatial import ConvexHull, QhullError
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Parameters ---
num_ramp_priority_patterns = 5
ramp_highlight_color = "red"
ramp_line_style = "--"

# --- Compute distance to ramp start for each pattern ---
ramp_df = mined_blocks.groupby("blast_id").agg(
    x_centroid=("x", "mean"),
    y_centroid=("y", "mean"),
    bench_level=("bench_level", "first"),
    pushback_x=("pushback_x", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    pushback_y=("pushback_y", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
).reset_index()

ramp_df["dist_to_ramp"] = (
    ((ramp_df["x_centroid"] - ramp_df["pushback_x"]) ** 2 +
     (ramp_df["y_centroid"] - ramp_df["pushback_y"]) ** 2) ** 0.5
)

# Get closest ramp patterns per campaign
blast_master["is_ramp_pattern"] = False
for campaign_id in blast_master["blast_campaign_id"].unique():
    campaign = blast_master[blast_master["blast_campaign_id"] == campaign_id]
    candidates = campaign.merge(ramp_df[["blast_id", "dist_to_ramp"]], on="blast_id", how="left")
    top_ramps = candidates.nsmallest(num_ramp_priority_patterns, "dist_to_ramp")
    blast_master.loc[blast_master["blast_id"].isin(top_ramps["blast_id"]), "is_ramp_pattern"] = True

# --- Redo sequence: ramp patterns first, then by pit development direction ---
def resequence_campaign(campaign_df):
    ramp_df = campaign_df[campaign_df["is_ramp_pattern"]].copy()
    other_df = campaign_df[~campaign_df["is_ramp_pattern"]].copy()

    # Apply directional sort to non-ramp patterns
    if pit_development_direction == "north_to_south":
        other_df = other_df.sort_values(by=["y_centroid", "x_centroid"], ascending=[True, True])
    elif pit_development_direction == "east_to_west":
        other_df = other_df.sort_values(by=["x_centroid", "y_centroid"], ascending=[False, True])
    elif pit_development_direction == "from_center":
        pit_center_x = campaign_df["x_centroid"].mean()
        pit_center_y = campaign_df["y_centroid"].mean()
        other_df["distance_from_center"] = (
            ((other_df["x_centroid"] - pit_center_x) ** 2 +
             (other_df["y_centroid"] - pit_center_y) ** 2) ** 0.5
        )
        other_df = other_df.sort_values(by="distance_from_center")

    combined = pd.concat([ramp_df, other_df], axis=0).reset_index(drop=True)
    combined["blast_sequence"] = range(1, len(combined) + 1)
    return combined

# Apply to all campaigns
updated_master = []
for cid in blast_master["blast_campaign_id"].unique():
    updated_master.append(resequence_campaign(blast_master[blast_master["blast_campaign_id"] == cid]))
blast_master = pd.concat(updated_master, axis=0).reset_index(drop=True)

# --- Dropdown UI for phase and bench ---
phases = sorted(blast_master["phase"].unique())
benches = sorted(blast_master["bench_rl"].unique(), reverse=True)

phase_dropdown = widgets.Dropdown(options=phases, description="Phase:")
bench_dropdown = widgets.Dropdown(options=benches, description="Bench RL:")

def plot_sequence(phase, bench):
    selected = blast_master[
        (blast_master["phase"] == phase) &
        (blast_master["bench_rl"] == bench)
    ]

    bench_blocks = mined_blocks[
        (mined_blocks["bench_rl"] == bench) &
        (mined_blocks["phase"] == phase)
    ]

    plt.figure(figsize=(12, 10))
    plt.scatter(bench_blocks["x"], bench_blocks["y"], s=5, c="lightgrey", label="Blocks")

    for _, row in selected.iterrows():
        pattern_blocks = bench_blocks[bench_blocks["blast_id"] == row["blast_id"]]
        points = pattern_blocks[["x", "y"]].values

        if len(points) < 3:
            continue

        try:
            hull = ConvexHull(points, qhull_options='QJ')
            hull_points = points[hull.vertices]
            if row["is_ramp_pattern"]:
                plt.plot(*hull_points.T, color=ramp_highlight_color, linestyle=ramp_line_style, linewidth=2)
            else:
                plt.plot(*hull_points.T, color="black", linewidth=1)

            # Sequence number
            plt.text(row["x_centroid"], row["y_centroid"], str(row["blast_sequence"]),
                     fontsize=8, ha="center", va="center", weight="bold")

        except QhullError:
            continue

    plt.title(f"Blast Pattern Sequence – PH{phase}_B{bench}")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.gca().set_aspect("equal")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def update_plot(*args):
    clear_output(wait=True)
    display(widgets.HBox([phase_dropdown, bench_dropdown]))
    plot_sequence(phase_dropdown.value, bench_dropdown.value)

phase_dropdown.observe(update_plot, names="value")
bench_dropdown.observe(update_plot, names="value")

# Initial display
update_plot()


### 📦 **Step 3.5: Advanced Ramp Geometry & Trim Logic**
This step will:
1. Redesign ramp patterns with:
    * Length based on bench_height * 10 + 3 * min_truck_width
    * Width = 3 * min_truck_width
2. Ensure directional continuity:
    * Follow the ramp direction
    * Fall back to pit_development_direction if no space ahead
3. Assign trim blasts:
    * All pit perimeters
    * Ramps on final walls will also be tagged is_trim_blast = True
#### 🧱 **Output:**
This step will update your blast pattern dataset (mined_blocks or preview_df) with:
* Realistic ramp geometry
* is_ramp_pattern (already exists)
* Updated blast_id and pattern_* columns
* Full set of is_trim_blast = True rows around walls

In [ ]:
# ======================================================================
# 🛠️ STEP 3.5: ADVANCED RAMP GEOMETRY & TRIM BLAST LOGIC
# ======================================================================

print("🔧 Adjusting ramp patterns with geometric logic...")

# --- Define ramp dimensions based on mining rules ---
ramp_length = (bench_height * 10) + (3 * min_truck_width)  # 1:10 gradient + turning room
ramp_width = 3 * min_truck_width
min_trim_width = 30  # meters (can optionally define in Step 1)

# Tag all blocks initially as not being ramp or trim
preview_df.loc[:, "is_trim_blast"] = False
preview_df.loc[:, "is_ramp_pattern"] = False

# --- Create a working copy of the bench-phase grid ---
ramp_blocks = preview_df.copy()

# --- Flag trim blasts based on proximity to pit edge ---
x_min, x_max = ramp_blocks["x"].min(), ramp_blocks["x"].max()
y_min, y_max = ramp_blocks["y"].min(), ramp_blocks["y"].max()

# Identify blocks close to final walls
trim_margin = min_trim_width
ramp_blocks["is_trim_blast"] = (
    (ramp_blocks["x"] <= x_min + trim_margin) |
    (ramp_blocks["x"] >= x_max - trim_margin) |
    (ramp_blocks["y"] <= y_min + trim_margin) |
    (ramp_blocks["y"] >= y_max - trim_margin)
)

# --- Redesign ramp patterns with appropriate size ---
# First, identify ramp start zones (assume pushback_x/y is defined)
# Filter for ramp blocks flagged previously
ramp_candidates = ramp_blocks[ramp_blocks["is_ramp_pattern"]].copy()

# Calculate direction vectors
def determine_ramp_orientation(row):
    dx = row["x"] - row["pushback_x"]
    dy = row["y"] - row["pushback_y"]
    if abs(dx) > abs(dy):
        return "E-W" if dx > 0 else "W-E"
    else:
        return "N-S" if dy > 0 else "S-N"

ramp_candidates["ramp_orientation"] = ramp_candidates.apply(determine_ramp_orientation, axis=1)

# Tag ramp blocks with proper bounding region
def tag_ramp_geometry(row):
    if row["ramp_orientation"] in ["N-S", "S-N"]:
        length_axis = "y"
        width_axis = "x"
    else:
        length_axis = "x"
        width_axis = "y"

    # Define bounding box for ramp
    half_width = ramp_width / 2
    length = ramp_length

    if length_axis == "x":
        x_min_ramp = row["x"] - length if row["ramp_orientation"] == "W-E" else row["x"]
        x_max_ramp = row["x"] if row["ramp_orientation"] == "W-E" else row["x"] + length
        y_min_ramp = row["y"] - half_width
        y_max_ramp = row["y"] + half_width
    else:
        y_min_ramp = row["y"] - length if row["ramp_orientation"] == "S-N" else row["y"]
        y_max_ramp = row["y"] if row["ramp_orientation"] == "S-N" else row["y"] + length
        x_min_ramp = row["x"] - half_width
        x_max_ramp = row["x"] + half_width

    return pd.Series({
        "x_min_ramp": x_min_ramp,
        "x_max_ramp": x_max_ramp,
        "y_min_ramp": y_min_ramp,
        "y_max_ramp": y_max_ramp,
    })

# Apply bounding box logic
ramp_geometry = ramp_candidates.apply(tag_ramp_geometry, axis=1)
ramp_candidates = pd.concat([ramp_candidates, ramp_geometry], axis=1)

# Flag ramp blocks within geometric window
for _, ramp in ramp_candidates.iterrows():
    in_ramp_zone = (
        (preview_df["x"] >= ramp["x_min_ramp"]) &
        (preview_df["x"] <= ramp["x_max_ramp"]) &
        (preview_df["y"] >= ramp["y_min_ramp"]) &
        (preview_df["y"] <= ramp["y_max_ramp"]) &
        (preview_df["bench_level"] == ramp["bench_level"]) &
        (preview_df["phase"] == ramp["phase"])
    )
    preview_df.loc[in_ramp_zone, "is_ramp_pattern"] = True

# --- Merge ramp and trim classifications ---
# Ramps on final wall are effectively trim blasts too
preview_df.loc[:, "is_trim_blast"] = preview_df["is_trim_blast"] | (
    preview_df["is_ramp_pattern"] & preview_df["is_trim_blast"]
)

print("✅ Ramp geometry and trim blast logic applied.")


## 📊 **Step 5: Updated Interactive Visualisation (Step 5 – Enhanced)**
This builds on your previous Step 5 but adds:
* Distinct styling for ramp and trim patterns
* Color-coded convex hulls
* Blast sequence number labels


In [ ]:
# ======================================================================
# 🧭 STEP 5: INTERACTIVE VISUALISATION – RAMPS & TRIMS
# ======================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.spatial import ConvexHull, QhullError
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dropdown options
phase_options = sorted(blast_master["phase"].unique())
bench_options = sorted(blast_master["bench_rl"].unique(), reverse=True)

phase_dropdown = widgets.Dropdown(options=phase_options, description="Phase:")
bench_dropdown = widgets.Dropdown(options=bench_options, description="Bench RL:")

def plot_patterns(phase, bench):
    clear_output(wait=True)
    display(widgets.HBox([phase_dropdown, bench_dropdown]))
    
    # Filter for selected campaign
    df_plot = blast_master[
        (blast_master["phase"] == phase) & 
        (blast_master["bench_rl"] == bench)
    ]
    blocks = mined_blocks[
        (mined_blocks["phase"] == phase) &
        (mined_blocks["bench_rl"] == bench)
    ]

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.scatter(blocks["x"], blocks["y"], s=5, c="lightgrey", label="Blocks")

    for _, row in df_plot.iterrows():
        pattern_blocks = blocks[blocks["blast_id"] == row["blast_id"]]
        if len(pattern_blocks) < 3:
            continue
        points = pattern_blocks[["x", "y"]].values

        try:
            hull = ConvexHull(points, qhull_options='QJ')
            hull_pts = points[hull.vertices]

            # Set color & style
            if row.get("is_ramp_pattern", False):
                edge_color = "red"
                line_style = "--"
                label = "Ramp"
            elif row.get("is_trim_blast", False):
                edge_color = "blue"
                line_style = ":"
                label = "Trim"
            else:
                edge_color = "black"
                line_style = "-"
                label = "Regular"

            ax.plot(*hull_pts.T, color=edge_color, linestyle=line_style, linewidth=1.5)

            # Draw blast sequence number
            ax.text(row["x_centroid"], row["y_centroid"], str(row["blast_sequence"]),
                    fontsize=8, ha="center", va="center", weight="bold")

        except QhullError:
            continue

    ax.set_title(f"Blast Patterns – Phase {phase}, Bench RL {bench}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_aspect("equal")
    ax.grid(True)
    ax.legend(["Ramp", "Trim", "Regular"], loc="upper right")
    plt.tight_layout()
    plt.show()

def update_plot(*args):
    plot_patterns(phase_dropdown.value, bench_dropdown.value)

phase_dropdown.observe(update_plot, names="value")
bench_dropdown.observe(update_plot, names="value")

# Initial plot
update_plot()


In [ ]:
# ======================================================================
# 🧱 ENHANCED: PLOT BLAST PATTERN POLYGONS (WITH TRIM BLAST LOGIC)
# ======================================================================

fig, ax = plt.subplots(figsize=(12, 10))

# Plot block centroids (light gray for context)
ax.scatter(preview_df["x"], preview_df["y"], c="lightgray", s=8, label="Blocks")

# Estimate half block size for polygon edge fitting
block_dx = mined_blocks["x"].diff().abs().median()
block_dy = mined_blocks["y"].diff().abs().median()

# Loop through each blast pattern
for blast_id, group in preview_df.groupby("blast_id"):
    if group.empty:
        continue

    # Approximate bounding box size
    pattern_width = group["x"].max() - group["x"].min()
    pattern_height = group["y"].max() - group["y"].min()
    is_trim = pattern_width < min_trim_width or pattern_height < min_trim_width

    # Expand blocks to their physical footprint
    expanded_points = []
    for _, row in group.iterrows():
        expanded_points.extend([
            [row["x"] - block_dx / 2, row["y"] - block_dy / 2],
            [row["x"] + block_dx / 2, row["y"] - block_dy / 2],
            [row["x"] + block_dx / 2, row["y"] + block_dy / 2],
            [row["x"] - block_dx / 2, row["y"] + block_dy / 2],
        ])
    expanded_points = np.array(expanded_points)

    try:
        if len(expanded_points) >= 3:
            hull = ConvexHull(expanded_points)
            polygon = patches.Polygon(
                expanded_points[hull.vertices],
                closed=True,
                edgecolor="red" if is_trim else "black",
                facecolor="none",
                linewidth=1.2 if is_trim else 1.0,
                linestyle="--" if is_trim else "-"
            )
            ax.add_patch(polygon)
        else:
            # Fallback bounding box
            xmin, ymin = group[["x", "y"]].min()
            xmax, ymax = group[["x", "y"]].max()
            polygon = patches.Rectangle(
                (xmin - block_dx / 2, ymin - block_dy / 2),
                (xmax - xmin) + block_dx,
                (ymax - ymin) + block_dy,
                edgecolor="red", facecolor="none", linewidth=1.2, linestyle="--"
            )
            ax.add_patch(polygon)
    except Exception as e:
        print(f"⚠️ Failed to draw polygon for {blast_id}: {e}")
        continue

ax.set_title(f"Blast Pattern Boundaries (with Trim Logic) – Bench RL {preview_bench}")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_aspect("equal")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ======================================================================
# 🟡 B: GOLD GRADE HEATMAP WITH BLAST BOUNDARIES
# ======================================================================

fig, ax = plt.subplots(figsize=(12, 10))

# Plot Au_grade as a heatmap
sc = ax.scatter(preview_df["x"], preview_df["y"], c=preview_df["Au_grade"],
                cmap="plasma", s=10, alpha=0.9)
plt.colorbar(sc, ax=ax, label="Au Grade (g/t)")

# Draw blast pattern polygons on top
for blast_id, group in preview_df.groupby("blast_id"):
    if group.shape[0] < 3:
        continue
    points = group[["x", "y"]].values
    try:
        hull = ConvexHull(points)
        polygon = patches.Polygon(points[hull.vertices], closed=True,
                                  edgecolor="black", facecolor="none", linewidth=1)
        ax.add_patch(polygon)
    except:
        continue

ax.set_title(f"Gold Grade Map + Blast Pattern Boundaries – Bench RL {preview_bench}")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_aspect("equal")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Display unique periods and their counts
unique_periods = mined_blocks["period"].value_counts()
print("Unique periods and their counts:")
print(unique_periods)

In [ ]:
print("Available bench levels:", mined_blocks["bench_rl"].unique())


# Visualise blast pattern grid for a single bench
preview_df = mined_blocks[mined_blocks["bench_level"] == preview_bench // bench_height]

plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=preview_df,
    x="x", y="y", hue="blast_id", palette="tab20", legend=False, s=10
)
plt.title(f"Blast Pattern Layout for Bench RL {preview_bench}")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)
plt.show()


## 🧠 **Step 4: Sequence Planning Logic**
Assign each pattern:
* phase and period based on:
	* Pushback logic from scheduling file
	* Tonnes-per-period limits
	* Ore availability for mill feed
	* Ramp and wall access logic (more below)

#### 💡 **Optional: You could enforce heuristics such as:**
* Prioritize ore-rich patterns earlier.
* Ensure inter-bench dependency (blast above must be fired before below).
* Reserve space for final wall and interim ramp zones (defined spatially or by buffer tags).

## 🧨 **Step 5: Blast Design Parameters per Pattern**
Estimate blast parameters per polygon:
* Burden & spacing based on UCS, RMR, grain_size
* Hole depth = bench_height + subdrill
* Explosive type from lith_code or alteration
* Powder factor, explosive mass, cost estimate

## 📦 **Step 6: Build Blast Master Table**
Create a table:
This becomes your blast master, feeding:
* Short-term plans
* Drill & blast plans
* Haulage/processing integration